In [1]:
# System
import os
import sys

os.environ["KERAS_BACKEND"] = "jax"
sys.path.append("../..")

In [2]:
# Setup
import json
from importlib import import_module
from pathlib import Path

import numpy as np
from keras import ops
from rich.table import Table

from src.models import GradientBoostedDecisionTree as BDT
from src.models import LearnableCutFlowParallel as LCF_PAR
from src.models import LearnableCutFlowSequential as LCF_SEQ
from src.models import MultiLayerPerceptron as MLP
from src.utils import Timer, load_model, print, to_numpy

In [3]:
# Parameters
rerun = False
n_runs = 10

# Dataset
dataset = "mock2"  # *
n_samples = 200000
seed = 42

# Model
centers = [-2, -1, -3]  # *
features = [r"$x_1$", r"$x_5$", r"$x_6$"]  # *

n_epochs = 200
batch_size = 512
verbose = 0

if not rerun and Path("results.json").exists():
    with open("results.json", "r") as f:
        results = json.load(f)
else:
    results = {}

In [4]:
# Dataset
module = import_module(f"src.datasets.{dataset}")
load_data = getattr(module, "load_data")
(x_train, y_train), (x_test, y_test) = load_data(n_samples, seed)

print(f"{x_train.shape=}")
print(f"{y_train.shape=}")
print(f"{x_test.shape=}")
print(f"{y_test.shape=}")

x_train.shape=(100000, 3)
y_train.shape=(100000, 1)
x_test.shape=(100000, 3)
y_test.shape=(100000, 1)


In [5]:
# Model: BDT
bdt_name = "bdt"
bdt_ckpt_paths = [Path(f"checkpoints/{bdt_name}@{i + 1}.pkl") for i in range(n_runs)]

bdt_ckpts = []
if rerun or not all(ckpt.exists() for ckpt in bdt_ckpt_paths):
    records = []
    for i in range(n_runs):
        print(f"Processing {bdt_name}@{i + 1}...")

        bdt = BDT(input_shape=x_train.shape, name=bdt_name)
        bdt.compile(optimizer="adam", loss="crossentropy")

        with Timer() as timer:
            bdt.fit(
                x_train,
                y_train.squeeze(),
                batch_size=batch_size,
                epochs=n_epochs,
                verbose=verbose,
            )
        records.append(timer.record)

        bdt.save(bdt_ckpt_paths[i])
        bdt_ckpts.append(load_model(bdt_ckpt_paths[i]))

    results[bdt_name] = {
        "training_time_mean": np.mean(records),
        "training_time_std": np.std(records),
    }

print(
    "Training time: "
    f"{results[bdt_name]['training_time_mean']:.2f} ± "
    f"{results[bdt_name]['training_time_std']:.2f} seconds"
)

KeyError: 'training_time_mean'

In [6]:
# Model: MLP
mlp_name = "mlp"
mlp_ckpt_paths = [Path(f"checkpoints/{mlp_name}@{i + 1}.keras") for i in range(n_runs)]

mlp_ckpts = []
if rerun or not all(ckpt.exists() for ckpt in mlp_ckpt_paths):
    records = []
    for i in range(n_runs):
        print(f"Processing {mlp_name}@{i + 1}...")

        mlp = MLP(x_train.shape, name=mlp_name)
        mlp.adapt(x_train)
        mlp.compile(optimizer="adam", loss="crossentropy")

        with Timer() as timer:
            mlp.fit(
                x_train,
                y_train,
                batch_size=batch_size,
                epochs=n_epochs,
                verbose=verbose,
            )
        records.append(timer.record)

        mlp.save(mlp_ckpt_paths[i])
        mlp_ckpts.append(load_model(mlp_ckpt_paths[i]))

    results[mlp_name] = {
        "training_time_mean": np.mean(records),
        "training_time_std": np.std(records),
    }

print(
    "Training time: "
    f"{results[mlp_name]['training_time_mean']:.2f} ± "
    f"{results[mlp_name]['training_time_std']:.2f} seconds"
)

Processing mlp@1...
Epoch 1/200
196/196 - 10s - 53ms/step - loss: 0.1782
Epoch 2/200
196/196 - 2s - 9ms/step - loss: 0.0819
Epoch 3/200
196/196 - 0s - 951us/step - loss: 0.0818
Epoch 4/200
196/196 - 0s - 941us/step - loss: 0.0816
Epoch 5/200
196/196 - 0s - 814us/step - loss: 0.0817
Epoch 6/200
196/196 - 0s - 812us/step - loss: 0.0815
Epoch 7/200
196/196 - 0s - 808us/step - loss: 0.0813
Epoch 8/200
196/196 - 0s - 805us/step - loss: 0.0812
Epoch 9/200
196/196 - 0s - 808us/step - loss: 0.0813
Epoch 10/200
196/196 - 0s - 845us/step - loss: 0.0813
Epoch 11/200
196/196 - 0s - 845us/step - loss: 0.0814
Epoch 12/200
196/196 - 0s - 845us/step - loss: 0.0812
Epoch 13/200
196/196 - 0s - 960us/step - loss: 0.0812
Epoch 14/200
196/196 - 0s - 1ms/step - loss: 0.0812
Epoch 15/200
196/196 - 0s - 869us/step - loss: 0.0812
Epoch 16/200
196/196 - 0s - 845us/step - loss: 0.0811
Epoch 17/200
196/196 - 0s - 853us/step - loss: 0.0810
Epoch 18/200
196/196 - 0s - 887us/step - loss: 0.0810
Epoch 19/200
196/196 

In [7]:
# Model: LCF(parallel)
lcf_par_name = "lcf_par"
lcf_par_ckpt_paths = [
    Path(f"checkpoints/{lcf_par_name}@{i + 1}.keras") for i in range(n_runs)
]

lcf_par_ckpts = []
if rerun or not all(ckpt.exists() for ckpt in lcf_par_ckpt_paths):
    records = []
    for i in range(n_runs):
        print(f"Processing {lcf_par_name}@{i + 1}...")

        lcf_par = LCF_PAR(x_train.shape, centers, features=features, name=lcf_par_name)
        lcf_par.adapt(x_train)
        lcf_par.compile(optimizer="adam", loss="crossentropy")

        with Timer() as timer:
            lcf_par.fit(
                x_train,
                y_train,
                batch_size=batch_size,
                epochs=n_epochs,
                verbose=verbose,
            )
        records.append(timer.record)

        lcf_par.save(lcf_par_ckpt_paths[i])
        lcf_par_ckpts.append(load_model(lcf_par_ckpt_paths[i]))

    results[lcf_par_name] = {
        "training_time_mean": np.mean(records),
        "training_time_std": np.std(records),
    }

print(
    "Training time: "
    f"{results[lcf_par_name]['training_time_mean']:.2f} ± "
    f"{results[lcf_par_name]['training_time_std']:.2f} seconds"
)

Processing lcf_par@1...
Epoch 1/200
196/196 - 3s - 18ms/step - loss: 0.4164
Epoch 2/200
196/196 - 1s - 6ms/step - loss: 0.3779
Epoch 3/200
196/196 - 0s - 2ms/step - loss: 0.3490
Epoch 4/200
196/196 - 0s - 1ms/step - loss: 0.3265
Epoch 5/200
196/196 - 0s - 2ms/step - loss: 0.3081
Epoch 6/200
196/196 - 0s - 2ms/step - loss: 0.2929
Epoch 7/200
196/196 - 0s - 1ms/step - loss: 0.2806
Epoch 8/200
196/196 - 0s - 1ms/step - loss: 0.2709
Epoch 9/200
196/196 - 0s - 1ms/step - loss: 0.2631
Epoch 10/200
196/196 - 0s - 1ms/step - loss: 0.2569
Epoch 11/200
196/196 - 0s - 2ms/step - loss: 0.2517
Epoch 12/200
196/196 - 0s - 2ms/step - loss: 0.2474
Epoch 13/200
196/196 - 0s - 2ms/step - loss: 0.2437
Epoch 14/200
196/196 - 0s - 2ms/step - loss: 0.2405
Epoch 15/200
196/196 - 0s - 2ms/step - loss: 0.2377
Epoch 16/200
196/196 - 0s - 2ms/step - loss: 0.2352
Epoch 17/200
196/196 - 0s - 2ms/step - loss: 0.2330
Epoch 18/200
196/196 - 0s - 2ms/step - loss: 0.2311
Epoch 19/200
196/196 - 0s - 1ms/step - loss: 0.2

In [8]:
# Model: LCF(sequential)
lcf_seq_name = "lcf_seq"
lcf_seq_ckpt_paths = [
    Path(f"checkpoints/{lcf_seq_name}@{i + 1}.keras") for i in range(n_runs)
]

lcf_seq_ckpts = []
if rerun or not all(ckpt.exists() for ckpt in lcf_seq_ckpt_paths):
    records = []
    for i in range(n_runs):
        print(f"Processing {lcf_seq_name}@{i + 1}...")

        lcf_seq = LCF_SEQ(x_train.shape, centers, features=features, name=lcf_seq_name)
        lcf_seq.adapt(x_train)
        lcf_seq.compile(optimizer="adam", loss="crossentropy")

        with Timer() as timer:
            lcf_seq.fit(
                x_train,
                y_train,
                batch_size=batch_size,
                epochs=n_epochs,
                verbose=verbose,
            )
        records.append(timer.record)

        lcf_seq.save(lcf_seq_ckpt_paths[i])
        lcf_seq_ckpts.append(load_model(lcf_seq_ckpt_paths[i]))

    results[lcf_seq_name] = {
        "training_time_mean": np.mean(records),
        "training_time_std": np.std(records),
    }

print(
    "Training time: "
    f"{results[lcf_seq_name]['training_time_mean']:.2f} ± "
    f"{results[lcf_seq_name]['training_time_std']:.2f} seconds"
)

Processing lcf_seq@1...
Epoch 1/200
196/196 - 4s - 20ms/step - loss: 0.1193
Epoch 2/200
196/196 - 1s - 6ms/step - loss: 0.1103
Epoch 3/200
196/196 - 0s - 1ms/step - loss: 0.1043
Epoch 4/200
196/196 - 0s - 1ms/step - loss: 0.1028
Epoch 5/200
196/196 - 0s - 1ms/step - loss: 0.1041
Epoch 6/200
196/196 - 0s - 1ms/step - loss: 0.1061
Epoch 7/200
196/196 - 0s - 1ms/step - loss: 0.1091
Epoch 8/200
196/196 - 0s - 1ms/step - loss: 0.1109
Epoch 9/200
196/196 - 0s - 2ms/step - loss: 0.1110
Epoch 10/200
196/196 - 0s - 1ms/step - loss: 0.1107
Epoch 11/200
196/196 - 0s - 1ms/step - loss: 0.1095
Epoch 12/200
196/196 - 0s - 1ms/step - loss: 0.1085
Epoch 13/200
196/196 - 0s - 1ms/step - loss: 0.1076
Epoch 14/200
196/196 - 0s - 1ms/step - loss: 0.1068
Epoch 15/200
196/196 - 0s - 1ms/step - loss: 0.1061
Epoch 16/200
196/196 - 0s - 1ms/step - loss: 0.1058
Epoch 17/200
196/196 - 0s - 1ms/step - loss: 0.1050
Epoch 18/200
196/196 - 0s - 1ms/step - loss: 0.1050
Epoch 19/200
196/196 - 0s - 1ms/step - loss: 0.1

In [ ]:
# Analysis: metrics
rerun = True
y_true = y_test

table = Table(title="Model Performance Comparison")
table.add_column("#", justify="center", style="cyan", no_wrap=True)
table.add_column("Model", style="magenta")
table.add_column("TP", justify="right", style="green")
table.add_column("FP", justify="right", style="red")
table.add_column("Accuracy", justify="right", style="blue")
table.add_column("Precision", justify="right", style="blue")
table.add_column("Significance", justify="right", style="yellow")
table.add_column("Time(s)", justify="right", style="yellow")

if rerun or not Path("resultsx10.json").exists():
    for ckpts in [bdt_ckpts, mlp_ckpts, lcf_par_ckpts, lcf_seq_ckpts]:
        print(f"Processing {ckpts[0].name}...")

        tp_list = []
        fp_list = []
        accuracy_list = []
        precision_list = []
        significance_list = []

        for i, ckpt in enumerate(ckpts):
            y_pred = ckpt.predict(x_test, batch_size=batch_size, verbose=0)
            y_pred = ops.all(y_pred > 0.5, axis=1, keepdims=True)

            tp = to_numpy(ops.sum((y_true == 1) & (y_pred == 1)))
            fp = to_numpy(ops.sum((y_true == 0) & (y_pred == 1)))
            tn = to_numpy(ops.sum((y_true == 0) & (y_pred == 0)))
            fn = to_numpy(ops.sum((y_true == 1) & (y_pred == 0)))

            accuracy = (tp + tn) / (tp + tn + fp + fn)
            precision = tp / (tp + fp)

            s = tp / (tp + tn + fp + fn) * 3000 * 1000 * 0.7644
            b = fp / (tp + tn + fp + fn) * 3000 * 1000 * 1.806 * 1e5
            significance = s / np.sqrt(b)

            tp_list.append(tp)
            fp_list.append(fp)
            accuracy_list.append(accuracy)
            precision_list.append(precision)
            significance_list.append(significance)

        tp_mean = np.mean(tp_list)
        fp_mean = np.mean(fp_list)
        accuracy_mean = np.mean(accuracy_list)
        precision_mean = np.mean(precision_list)
        significance_mean = np.mean(significance_list)

        tp_std = np.std(tp_list)
        fp_std = np.std(fp_list)
        accuracy_std = np.std(accuracy_list)
        precision_std = np.std(precision_list)
        significance_std = np.std(significance_list)

        results[ckpts[0].name].update(
            {
                "tp_mean": tp_mean.tolist(),
                "tp_std": tp_std.tolist(),
                "fp_mean": fp_mean.tolist(),
                "fp_std": fp_std.tolist(),
                "accuracy_mean": accuracy_mean.tolist(),
                "accuracy_std": accuracy_std.tolist(),
                "precision_mean": precision_mean.tolist(),
                "precision_std": precision_std.tolist(),
                "significance_mean": significance_mean.tolist(),
                "significance_std": significance_std.tolist(),
            }
        )

        with open("resultsx10.json", "w") as f:
            json.dump(results, f, indent=4)

for i, (name, metrics) in enumerate(results.items()):
    tp_mean = metrics["tp_mean"]
    tp_std = metrics["tp_std"]
    fp_mean = metrics["fp_mean"]
    fp_std = metrics["fp_std"]
    accuracy_mean = metrics["accuracy_mean"]
    accuracy_std = metrics["accuracy_std"]
    precision_mean = metrics["precision_mean"]
    precision_std = metrics["precision_std"]
    significance_mean = metrics["significance_mean"]
    significance_std = metrics["significance_std"]
    training_time_mean = metrics["training_time_mean"]
    training_time_std = metrics["training_time_std"]

    table.add_row(
        str(i + 1),
        name,
        f"{tp_mean:.0f}\n± {tp_std:.0f}",
        f"{fp_mean:.0f}\n± {fp_std:.0f}",
        f"{accuracy_mean:.4f}\n± {accuracy_std:.4f}",
        f"{precision_mean:.4f}\n± {precision_std:.4f}",
        f"{significance_mean:.4f}\n± {significance_std:.4f}",
        f"{training_time_mean:.2f}\n± {training_time_std:.2f}",
    )

print(table)

Processing bdt...


Processing mlp...
Processing lcf_par...
Processing lcf_seq...
                         Model Performance Comparison                         
┏━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ # ┃ Model   ┃    TP ┃   FP ┃ Accuracy ┃ Precision ┃ Significance ┃ Time(s) ┃
┡━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ 1 │ bdt     │ 48251 │ 1493 │   0.9690 │    0.9700 │      12.3027 │   16.37 │
│   │         │   ± 0 │  ± 0 │ ± 0.0000 │  ± 0.0000 │     ± 0.0000 │  ± 0.92 │
│ 2 │ mlp     │ 48250 │ 1471 │   0.9692 │    0.9704 │      12.4125 │   69.08 │
│   │         │  ± 99 │ ± 95 │ ± 0.0002 │  ± 0.0018 │     ± 0.3817 │  ± 5.15 │
│ 3 │ lcf_par │ 27115 │  154 │   0.7710 │    0.9943 │      21.4914 │   60.70 │
│   │         │  ± 33 │  ± 0 │ ± 0.0003 │  ± 0.0000 │     ± 0.0154 │  ± 0.88 │
│ 4 │ lcf_seq │ 40570 │ 1279 │   0.8943 │    0.9694 │      11.1749 │   63.00 │
│   │         │  ± 22 │  ± 6 │ ± 0.0002 │  ± 0.0001 │     ± 0.0203 │ 